# GGUF руками: от отдельных байтов до тензора

Мы создадим корректный GGUF v3 без `llama.cpp`, NumPy и готового сериализатора. В файле будут две записи метаданных и один тензор F32 формы 2×2. Ожидаемый размер — **176 байт**, начало данных тензора — смещение **160**.

In [ ]:
from dataclasses import dataclass
import hashlib
import io
import struct

GGUF_VERSION = 3
GGUF_TYPE_UINT32 = 4
GGUF_TYPE_STRING = 8
GGML_TYPE_F32 = 0
ALIGNMENT = 32

def u32(value: int) -> bytes:
    return struct.pack('<I', value)

def u64(value: int) -> bytes:
    return struct.pack('<Q', value)

def gguf_string(value: str) -> bytes:
    raw = value.encode('utf-8')
    return u64(len(raw)) + raw

## 1. Заголовок и метаданные

Все целые числа записываются младшим байтом вперёд. После сигнатуры идут версия, число тензоров и число пар метаданных. Тип значения хранится отдельно от самого значения.

In [ ]:
blob = bytearray()
blob += b'GGUF'
blob += u32(GGUF_VERSION)
blob += u64(1)  # tensor_count
blob += u64(2)  # metadata_kv_count

blob += gguf_string('general.alignment')
blob += u32(GGUF_TYPE_UINT32)
blob += u32(ALIGNMENT)

blob += gguf_string('general.name')
blob += u32(GGUF_TYPE_STRING)
blob += gguf_string('booklm.demo')

assert len(blob) == 100
len(blob)

## 2. Описание тензора и выравнивание

Смещение тензора в описании отсчитывается не от начала файла, а от начала области данных. Для первого тензора оно равно нулю. Саму область данных выравниваем до 32 байт.

In [ ]:
blob += gguf_string('weight')
blob += u32(2)        # n_dimensions
blob += u64(2) + u64(2)
blob += u32(GGML_TYPE_F32)
blob += u64(0)        # relative tensor offset

assert len(blob) == 146
padding = (-len(blob)) % ALIGNMENT
blob += bytes(padding)
tensor_data_offset = len(blob)
assert padding == 14
assert tensor_data_offset == 160

expected_values = (1.0, -2.0, 3.5, 0.25)
blob += struct.pack('<4f', *expected_values)
assert len(blob) == 176
print('file bytes:', len(blob), 'tensor data offset:', tensor_data_offset)

In [ ]:
def hex_rows(data: bytes, width: int = 16) -> str:
    rows = []
    for offset in range(0, len(data), width):
        chunk = data[offset:offset + width]
        rows.append(f'{offset:04x}: ' + ' '.join(f'{b:02x}' for b in chunk))
    return '\n'.join(rows)

print(hex_rows(blob[:64]))
print('...')
print(hex_rows(blob[144:]))

## 3. Независимый читатель

Читатель ниже ничего не знает о переменных, использованных при записи. Он заново разбирает длины строк, типы метаданных и описание тензора.

In [ ]:
def read_exact(stream: io.BytesIO, size: int) -> bytes:
    value = stream.read(size)
    if len(value) != size:
        raise ValueError('unexpected end of file')
    return value

def read_u32(stream: io.BytesIO) -> int:
    return struct.unpack('<I', read_exact(stream, 4))[0]

def read_u64(stream: io.BytesIO) -> int:
    return struct.unpack('<Q', read_exact(stream, 8))[0]

def read_string(stream: io.BytesIO) -> str:
    size = read_u64(stream)
    return read_exact(stream, size).decode('utf-8')

@dataclass
class TensorInfo:
    name: str
    dimensions: tuple
    ggml_type: int
    relative_offset: int

def read_demo_gguf(data: bytes):
    stream = io.BytesIO(data)
    if read_exact(stream, 4) != b'GGUF':
        raise ValueError('not a GGUF file')
    version = read_u32(stream)
    tensor_count = read_u64(stream)
    metadata_count = read_u64(stream)
    metadata = {}
    for _ in range(metadata_count):
        key = read_string(stream)
        value_type = read_u32(stream)
        if value_type == GGUF_TYPE_UINT32:
            value = read_u32(stream)
        elif value_type == GGUF_TYPE_STRING:
            value = read_string(stream)
        else:
            raise ValueError(f'unsupported metadata type: {value_type}')
        metadata[key] = value
    tensors = []
    for _ in range(tensor_count):
        name = read_string(stream)
        n_dimensions = read_u32(stream)
        dimensions = tuple(read_u64(stream) for _ in range(n_dimensions))
        tensors.append(TensorInfo(name, dimensions, read_u32(stream), read_u64(stream)))
    alignment = metadata.get('general.alignment', 32)
    data_start = (stream.tell() + alignment - 1) // alignment * alignment
    return version, metadata, tensors, data_start

version, metadata, tensors, data_start = read_demo_gguf(bytes(blob))
info = tensors[0]
start = data_start + info.relative_offset
values = struct.unpack('<4f', blob[start:start + 16])

assert version == 3
assert metadata == {'general.alignment': 32, 'general.name': 'booklm.demo'}
assert info == TensorInfo('weight', (2, 2), GGML_TYPE_F32, 0)
assert data_start == 160
assert values == expected_values
metadata, info, values

In [ ]:
sha256 = hashlib.sha256(blob).hexdigest()
manifest = {
    'id': 'booklm-gguf-byte-demo-v1',
    'parents': {},                     # байты заданы руками: родителей нет
    'format': 'gguf',
    'format_version': version,
    'file_bytes': len(blob),
    'tensor_data_offset': data_start,
    'sha256': sha256,
}
assert len(sha256) == 64
manifest

## Задания

1. До запуска рассчитайте новый размер файла, если имя тензора увеличить на один байт. Учтите выравнивание.
2. Добавьте второй тензор и объясните, от какой точки отсчитывается его смещение.
3. Намеренно запишите `version` через `'>I'`. Какой результат получит читатель и почему сигнатура останется правильной?
4. Реализуйте чтение массива метаданных. Сначала выпишите его рекурсивный контракт.
5. Сопоставьте поля учебного файла со структурами `gguf_context` и `gguf_tensor_info` в исходном коде llama.cpp.